# TF-Serving of focus and circuli detectors in a Docker container for production

- This notebook describes the process of preparing the focus detector and circuli detector (their best performing models) for production

- TensorFlow Serving implements a server that processes incoming requests and forwards them to a model. This server could be running somewhere, locally or in a cloud provider

- Nowadays it is common to pack such server and all its dependencies into a package, configure it and deploy this package as a whole

- Containers are used for the creation of deployable artefacts. Such container will include your service and all its dependencies. You only need to install a thin layer that runs your container on an operation system

In [1]:
import os
import sys
import pandas as pd
import numpy as np

#os.chdir("c:/Users/dmp/Dropbox/dmp/clients/marine scotland/scale_image_analysis/code/models/yolov3-tf2_rectangular")
os.chdir("c:/Users/dmp/Dropbox/dmp/clients/marine scotland/scale_image_analysis/code/models/yolov3-tf2")
#sys.path.append('c:/Users/dmp/Dropbox/dmp/clients/marine scotland/scale_image_analysis/code/')
#from tools.utils import del_all_flags

%load_ext autoreload

## 1. Save and export the (best) focus and circuli detector models for TF-serving

In [ ]:
python export_tfserving.py ^
    --output serving/yolov3/2/

In [2]:
%run "export_tfserving.py"\
    --output serving/yolov3/2/

I0114 22:16:05.445760 19748 export_tfserving.py:32] weights loaded


Instructions for updating:
If using Keras pass *_constraint arguments to layers.


W0114 22:16:37.723686 19748 deprecation.py:506] From C:\Users\dmp\anaconda3\envs\yolov3-tf2-gpu\lib\site-packages\tensorflow_core\python\ops\resource_variable_ops.py:1786: calling BaseResourceVariable.__init__ (from tensorflow.python.ops.resource_variable_ops) with constraint is deprecated and will be removed in a future version.
Instructions for updating:
If using Keras pass *_constraint arguments to layers.


INFO:tensorflow:Assets written to: serving/yolov3/2/assets


I0114 22:16:44.088564 19748 builder_impl.py:775] Assets written to: serving/yolov3/2/assets
I0114 22:16:47.340476 19748 export_tfserving.py:35] model saved to: serving/yolov3/2/
I0114 22:17:01.732087 19748 export_tfserving.py:39] {'yolo_nms_1': TensorSpec(shape=(None, 100), dtype=tf.float32, name='yolo_nms_1'), 'yolo_nms_2': TensorSpec(shape=(None, 100), dtype=tf.float32, name='yolo_nms_2'), 'yolo_nms_3': TensorSpec(shape=(None,), dtype=tf.int32, name='yolo_nms_3'), 'yolo_nms': TensorSpec(shape=(None, 100, 4), dtype=tf.float32, name='yolo_nms')}
I0114 22:17:01.742086 19748 export_tfserving.py:42] classes loaded


UnknownError: 2 root error(s) found.
  (0) Unknown:  Failed to get convolution algorithm. This is probably because cuDNN failed to initialize, so try looking to see if a warning log message was printed above.
	 [[{{node StatefulPartitionedCall/yolov3/yolo_darknet/conv2d/Conv2D}}]]
	 [[StatefulPartitionedCall/yolov3/yolo_nms/Reshape_9/_14]]
  (1) Unknown:  Failed to get convolution algorithm. This is probably because cuDNN failed to initialize, so try looking to see if a warning log message was printed above.
	 [[{{node StatefulPartitionedCall/yolov3/yolo_darknet/conv2d/Conv2D}}]]
0 successful operations.
0 derived errors ignored. [Op:__inference_signature_wrapper_92848]

Function call stack:
signature_wrapper -> signature_wrapper


In [ ]:
%run  "detect_multiple.py" \
	--classes "D:/MSS_Scales_depot/circuli_detection/inputs/main_analysis/scale_transects_label.names" \
	--num_classes 1 \
    --img_width 3904 \
    --img_height 64 \
	--weights "D:/MSS_Scales_depot/circuli_detection/outputs/main_analysis/rctYolo_64x3904_padded_10EarlStop_1e-3LR_LRScheduling/checkpoints/yolov3_train_26.tf" \
    --tfrecord "D:/MSS_Scales_depot/circuli_detection/inputs/main_analysis/tfrecords/scale_transects_main_resized_padded_64x3904_valid.tfrecord"\
    --groundtruths_dir "D:/MSS_Scales_depot/circuli_detection/outputs/main_analysis/rctYolo_64x3904_padded_10EarlStop_1e-3LR_LRScheduling/results/validation_set_30ScrThresh/groundtruths" \
    --detections_dir "D:/MSS_Scales_depot/circuli_detection/outputs/main_analysis/rctYolo_64x3904_padded_10EarlStop_1e-3LR_LRScheduling/results/validation_set_30ScrThresh/detections" \
    --detection_img_dir "D:/MSS_Scales_depot/circuli_detection/outputs/main_analysis/rctYolo_64x3904_padded_10EarlStop_1e-3LR_LRScheduling/results/validation_set_30ScrThresh/detections_images" \
    --plot_images True \
    --yolo_score_threshold 0.3 \
    --yolo_iou_threshold 0.5 \
    --yolo_max_boxes 150

## 2. Build a Docker image for Tensorlow Serving to comprise the SSCD models

- Assumes Docker has been installed locally